# 4. Scaled Dot-Product Attention с нуля

**Цель:** Реализовать механизм внимания с нуля на PyTorch, понять Query/Key/Value, визуализировать матрицу внимания и сравнить с оптимизированной реализацией.

---

In [ ]:
# === Setup: imports and configuration ===
import sys, os, logging, math
# Logging to trace tensor shapes at each forward step
LOG_LEVEL = os.getenv("LOG_LEVEL", "DEBUG")
logging.basicConfig(level=getattr(logging, LOG_LEVEL), format="%(asctime)s [%(levelname)s] %(name)s: %(message)s", stream=sys.stderr)
log = logging.getLogger("attention")

# Torch for tensor ops; nn.functional has softmax + optimized attention
import torch
import torch.nn as nn
import torch.nn.functional as F
# NumPy for entropy and distribution stats
import numpy as np
# Matplotlib for attention heatmaps and score histograms
import matplotlib.pyplot as plt

# Prefer MPS (Apple Silicon Metal) over CPU for fast matrix multiply
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
log.info("Using device: %s", device)


## 4.1 Постановка задачи: зачем нужно внимание?

**Проблема RNN/LSTM:**
- Скрытое состояние сжимает всю информацию о последовательности в один вектор
- При длинных последовательностях информация "забывается" (vanishing gradient)
- Нет прямого доступа к произвольным позициям входа

**Идея внимания (Bahdanau, 2014):**
- На каждом шаге декодирования "смотрим" на все позиции входа
- Решаем, какие части входа наиболее релевантны для текущего шага
- Взвешенная сумма скрытых состояний энкодера

**Scaled Dot-Product Attention (Vaswani et al., 2017):**
- Query (запрос) — что мы ищем
- Key (ключ) — что у нас есть
- Value (значение) — информация, которую мы хотим извлечь

## 4.2 Query, Key, Value — интуиция и математика

**Аналогия (поиск в словаре):**
- **Query** — слово, которое мы ищем
- **Key** — заголовки словарных статей
- **Value** — содержимое статей
- **Score** (Q·Kᵀ) — насколько хорошо запрос соответствует каждому ключу
- **Softmax** — нормализация scores в распределение вероятностей
- **Output** = взвешенная сумма Values по этим весам

**Математика:**
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

Зачем делить на $\sqrt{d_k}$? Чтобы дисперсия scores оставалась ~1, и softmax не уходил в режим "почти one-hot" или "почти uniform" при большой размерности.

In [ ]:
log.debug("Implementing scaled dot-product attention from scratch")

def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Scaled Dot-Product Attention.
    
    Args:
        Q: (batch, seq_len_q, d_k)
        K: (batch, seq_len_k, d_k)
        V: (batch, seq_len_k, d_v)
        mask: (batch, seq_len_q, seq_len_k) or (seq_len_q, seq_len_k) — True where masked
    
    Returns:
        output: (batch, seq_len_q, d_v)
        attention_weights: (batch, seq_len_q, seq_len_k)
    """
    d_k = Q.size(-1)
    # === 1. Dot-Product Scores: Q @ K^T ===
    # Each element [i,j] = dot product between Q[i] and K[j].
    # Higher value = stronger query-key match in d_k-dimensional space.
    # Complexity: O(n^2 * d_k) — n^2 pairs, each requiring d_k multiplications.
    scores = Q @ K.transpose(-2, -1)  # (batch, seq_len_q, seq_len_k)
    # === 2. Scaling: / sqrt(d_k) ===
    # Why: if Q,K ~ N(0,1), then Var(Q*K) = d_k. At d_k=512 variance ~512.
    # Softmax on huge-variance inputs becomes near one-hot -> gradients vanish.
    # Dividing by sqrt(d_k) normalises variance back to ~1: Var(/sqrt(d_k)) = 1.
    scores = scores / math.sqrt(d_k)
    log.debug("Scores shape: %s, d_k=%d", scores.shape, d_k)
    # === 3. Masking ===
    # Replace padding positions (mask == 0) with -inf.
    # Why -inf: softmax(e^{-inf}) = 0 (zero weight); softmax(e^0) = 1 (distorts).
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
        log.debug("Mask applied, masked positions: %d", mask.eq(0).sum().item())
    # === 4. Softmax over keys ===
    # p_j = exp(s_j) / sum_k exp(s_k) — each row sums to 1.
    # These weights encode "how much attention" Q[i] pays to K[j].
    attention_weights = F.softmax(scores, dim=-1)
    log.debug("Attention weights shape: %s", attention_weights.shape)
    # === 5. Weighted Sum: A @ V ===
    # output[i] = sum_j A[i,j] * V[j] — convex combination of Values.
    # Complexity: O(n^2 * d_v) — second quadratic term in n.
    output = attention_weights @ V  # (batch, seq_len_q, d_v)
    log.debug("Output shape: %s", output.shape)
    return output, attention_weights


## Почему Scaled Dot-Product: анализ дисперсии

Пусть $Q$ и $K$ — случайные векторы с независимыми компонентами, имеющими нулевое среднее и единичную дисперсию. Тогда каждый элемент матрицы $QK^T$ является суммой $d_k$ произведений:

$$(QK^T)_{ij} = \sum_{m=1}^{d_k} Q_{im} K_{jm}$$

Каждое слагаемое имеет среднее 0 и дисперсию 1, поэтому:

$$\mathbb{E}[(QK^T)_{ij}] = 0, \quad \mathbb{Var}[(QK^T)_{ij}] = d_k$$

**Проблема:** при больших $d_k$ дисперсия scores растёт линейно. Softmax от значений с большой дисперсией становится «острым» (peaky) — одно значение near 1, остальные near 0. Градиенты в таких регионах экспоненциально малы.

**Решение:** деление на $\sqrt{d_k}$ нормализует дисперсию обратно к 1:

$$\mathbb{Var}\left[\frac{(QK^T)_{ij}}{\sqrt{d_k}}\right] = 1$$

Теперь softmax получает входы с единичной дисперсией, градиенты остаются «здоровыми», и обучение стабильно.


In [ ]:
# === Visualise score distribution BEFORE vs AFTER scaling ===
# Empirically shows how sqrt(d_k) normalises variance from d_k back to 1
log.debug("Visualising score distribution before/after scaling")

d_k = 64
# Generate Q and K as in real attention: (1, N, d_k) -> scores (N, N)
seq_len = 100  # 100 tokens -> 10 000 score pairs
Q = torch.randn(1, seq_len, d_k)
K = torch.randn(1, seq_len, d_k)

# Dot-product scores before and after scaling
scores = Q @ K.transpose(-2, -1)  # (1, 100, 100)
scores_before = scores.flatten()  # all 10 000 pairwise values
scores_after = scores_before / math.sqrt(d_k)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# BEFORE scaling — wide spread, variance ~ d_k
ax = axes[0]
ax.hist(scores_before.numpy(), bins=50, alpha=0.7, color="coral")
ax.axvline(scores_before.mean().item(), color="red", linestyle="--", label=f"mean={scores_before.mean():.2f}")
ax.axvline(scores_before.mean().item() + scores_before.std().item(), color="red", linestyle=":", label=f"std={scores_before.std():.2f}")
ax.axvline(scores_before.mean().item() - scores_before.std().item(), color="red", linestyle=":")
ax.set_title(f"BEFORE scaling (d_k={d_k})\nVar ≈ {scores_before.var():.1f} (≈ d_k)")
ax.set_xlabel("Score value")
ax.set_ylabel("Frequency")
ax.legend()
ax.grid(True, alpha=0.3)

# AFTER scaling — narrow spread, variance ~ 1
ax = axes[1]
ax.hist(scores_after.numpy(), bins=50, alpha=0.7, color="steelblue")
ax.axvline(scores_after.mean().item(), color="blue", linestyle="--", label=f"mean={scores_after.mean():.2f}")
ax.axvline(scores_after.mean().item() + scores_after.std().item(), color="blue", linestyle=":", label=f"std={scores_after.std():.2f}")
ax.axvline(scores_after.mean().item() - scores_after.std().item(), color="blue", linestyle=":")
ax.set_title(f"AFTER scaling (+{chr(247)}{chr(8730)}{d_k})\nVar ≈ {scores_after.var():.1f} (≈ 1)")
ax.set_xlabel("Score value")
ax.set_ylabel("Frequency")
ax.legend()
ax.grid(True, alpha=0.3)

plt.suptitle("Distribution of Attention Scores Before and After Scaling", fontsize=14)
plt.tight_layout()
plt.show()
log.info("Score distribution visualisation complete")


In [ ]:
log.debug("Testing attention on simple random data")

# 2 batches, 4 tokens, dim=8
batch_size, seq_len, d_k, d_v = 2, 4, 8, 8
# Random Q/K/V from N(0,1) — verify function runs shape-correctly
Q = torch.randn(batch_size, seq_len, d_k)
K = torch.randn(batch_size, seq_len, d_k)
V = torch.randn(batch_size, seq_len, d_v)

output, attn_weights = scaled_dot_product_attention(Q, K, V)

# Verify dimensions
print(f"Q shape:     {Q.shape}")
print(f"K shape:     {K.shape}")
print(f"V shape:     {V.shape}")
print(f"Output shape:{output.shape}")
print(f"Attention weights shape: {attn_weights.shape}")

# Inspect weights for first batch item
print(f"\nAttention weights (batch 0):\n{attn_weights[0].detach().numpy().round(3)}")

# Key invariant: softmax guarantees every row sums to 1.0
print(f"\nRow sums (should be ~1.0): {attn_weights[0].sum(dim=-1).numpy().round(4)}")

log.info("Basic attention test passed")


## Вывод градиентов через attention

Рассмотрим проход внимания: $S = QK^T / \sqrt{d_k}$, $A = \text{softmax}(S)$, $O = AV$.

**Градиент по V:**
$$\frac{\partial L}{\partial V} = A^T \frac{\partial L}{\partial O}$$

**Градиент по A (весам внимания):**
$$\frac{\partial L}{\partial A} = \frac{\partial L}{\partial O} V^T$$

**Градиент по S (до softmax):**
$$\frac{\partial L}{\partial S} = A \odot \left( \frac{\partial L}{\partial A} - \mathbf{1} \left( A \odot \frac{\partial L}{\partial A} \right) \right)$$

где $\odot$ — поэлементное умножение, а $\mathbf{1}$ — матрица из единиц. Это локальная производная softmax: $dA/dS = A \cdot (\delta_{ij} - A_j)$.

**Градиент по Q и K:**
$$\frac{\partial L}{\partial Q} = \frac{1}{\sqrt{d_k}} \frac{\partial L}{\partial S} K, \quad
\frac{\partial L}{\partial K} = \frac{1}{\sqrt{d_k}} \frac{\partial L}{\partial S}^T Q$$

Весь градиентный поток выражается через матричные умножения — никаких рекуррентных зависимостей, что позволяет полностью распараллелить вычисления.


## 4.3 Визуализация матрицы внимания

Heatmap — основной инструмент анализа. Показывает, какие токены "смотрят" на какие.

In [ ]:
log.debug("Visualizing attention heatmap")

# Synthetic data with controlled structure
seq_len = 6
d_k = 4

Q = torch.randn(1, seq_len, d_k)
K = torch.randn(1, seq_len, d_k)
V = torch.randn(1, seq_len, d_k)

# Induce a pattern: first 3 queries attend to last 3 keys
Q[:, :3, :] = 0
K[:, 3:, :] = 0
Q[:, :3, :] = Q[:, 3:, :] * 2  # amplify query-key similarity

output, attn_weights = scaled_dot_product_attention(Q, K, V)

# Heatmap: rows=queries, columns=keys, colour=attention weight
plt.figure(figsize=(8, 6))
plt.imshow(attn_weights[0].detach().numpy(), cmap='Blues')
plt.colorbar(label='Attention weight')
plt.xlabel('Key positions')
plt.ylabel('Query positions')
plt.title('Attention Heatmap')
plt.xticks(range(seq_len))
plt.yticks(range(seq_len))
for i in range(seq_len):
    for j in range(seq_len):
        val = attn_weights[0, i, j].item()
        plt.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=9)
plt.tight_layout()
plt.show()
log.info("Attention heatmap plotted")


## 4.4 Маскировка: Padding Mask

В реальных данных последовательности имеют разную длину. Мы паддим (дополняем) короткие последовательности специальным PAD-токеном и маскируем его, чтобы он не влиял на attention.

In [ ]:
log.debug("Demonstrating padding mask")

# 2 sequences of different lengths in one batch
batch, seq_len, d_k = 2, 5, 8
lengths = torch.tensor([3, 5])

# === Build mask ===
# mask[i,j] = True if j < lengths[i] (token is valid, not padding)
# Broadcast: torch.arange(5) -> (1,5); lengths.unsqueeze(1) -> (2,1)
# Result: (2,5) — True for positions before sequence end
mask = torch.arange(seq_len).unsqueeze(0) < lengths.unsqueeze(1)  # (batch, seq_len)
# Add head dim for broadcasting against scores: (batch, 1, seq_len)
mask = mask.unsqueeze(1)  # (batch, 1, seq_len)
print(f"Padding mask:\n{mask}")

Q = torch.randn(batch, seq_len, d_k)
K = torch.randn(batch, seq_len, d_k)
V = torch.randn(batch, seq_len, d_k)

output, attn_weights = scaled_dot_product_attention(Q, K, V, mask=mask)

# Two heatmaps: first seq (len 3) has positions 3,4 masked (zero weight)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for i in range(2):
    ax = axes[i]
    im = ax.imshow(attn_weights[i].detach().numpy(), cmap='Blues')
    ax.set_title(f'Sequence {i+1} (length {lengths[i].item()})')
    ax.set_xlabel('Key positions')
    ax.set_ylabel('Query positions')
    plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()
log.info("Padding mask visualization complete")


## 4.5 Демонстрация на синтетических данных: поиск соответствий

Создадим задачу, где внимание естественно возникает: нужно найти, какие элементы одной последовательности соответствуют элементам другой.

In [ ]:
log.debug("Running attention demo on synthetic matching task")

torch.manual_seed(42)
n_patterns = 4
d_k = 16
seq_len = 6

# Base patterns with additive noise
patterns = torch.randn(n_patterns, d_k)

# Q and K sample from same patterns — attention should match similar items
Q = patterns[torch.randint(0, n_patterns, (seq_len,))] + 0.1 * torch.randn(seq_len, d_k)
K = patterns[torch.randint(0, n_patterns, (seq_len,))] + 0.1 * torch.randn(seq_len, d_k)
V = K.clone()  # identity attention: output = weighted keys

Q, K, V = Q.unsqueeze(0), K.unsqueeze(0), V.unsqueeze(0)

output, attn_weights = scaled_dot_product_attention(Q, K, V)

# Expect block-diagonal structure (same patterns match each other)
plt.figure(figsize=(8, 6))
plt.imshow(attn_weights[0].detach().numpy(), cmap='Blues')
plt.colorbar(label='Attention weight')
plt.title('Attention: Pattern Matching')
plt.xlabel('Key position')
plt.ylabel('Query position')
for i in range(seq_len):
    for j in range(seq_len):
        val = attn_weights[0, i, j].item()
        plt.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=8)
plt.tight_layout()
plt.show()
log.info("Pattern matching attention demo complete")


## 4.6 Сравнение с оптимизированной реализацией PyTorch

PyTorch предоставляет `torch.nn.functional.scaled_dot_product_attention`. Сравним результаты.

In [ ]:
log.debug("Comparing manual implementation with torch's optimized version")

torch.manual_seed(123)
batch, seq_len, d_k = 4, 8, 32
Q = torch.randn(batch, seq_len, d_k)
K = torch.randn(batch, seq_len, d_k)
V = torch.randn(batch, seq_len, d_k)

# Our manual implementation (Q@K, scaling, softmax, A@V)
output_manual, attn_manual = scaled_dot_product_attention(Q, K, V)

# PyTorch optimised: uses Flash Attention / Memory-Efficient Attention
# but is mathematically equivalent to our step-by-step version
output_torch = F.scaled_dot_product_attention(Q, K, V)

# Should match within float32 machine precision
diff = (output_manual - output_torch).abs().max().item()
print(f"Max difference: {diff:.2e}")
print(f"Outputs match: {torch.allclose(output_manual, output_torch, atol=1e-6)}")
log.info("Comparison with F.scaled_dot_product_attention: max_diff=%.2e", diff)


In [ ]:
# === Performance Benchmark ===
import time

log.debug("Running performance benchmark")

# Realistic sizes: large batch, moderate sequence length
batch, seq_len, d_k = 16, 128, 64
# Place tensors on target device for realistic timing
Q = torch.randn(batch, seq_len, d_k, device=device)
K = torch.randn(batch, seq_len, d_k, device=device)
V = torch.randn(batch, seq_len, d_k, device=device)

def bench(fn, name, n_runs=20):
    # warmup: cache population and JIT compilation
    for _ in range(3):
        fn()
    if device.type == 'mps':
        torch.mps.synchronize()
    
    start = time.perf_counter()
    for _ in range(n_runs):
        fn()
    if device.type == 'mps':
        torch.mps.synchronize()
    elapsed = (time.perf_counter() - start) / n_runs
    print(f"{name:30s}: {elapsed*1000:.3f} ms")
    log.debug("Benchmark %s: %.3f ms", name, elapsed*1000)
    return elapsed

# Manual vs PyTorch optimised speed comparison
manual_time = bench(lambda: scaled_dot_product_attention(Q, K, V), "Manual attention")
torch_time = bench(lambda: F.scaled_dot_product_attention(Q, K, V), "F.scaled_dot_product_attention")
print(f"Speedup: {manual_time / torch_time:.1f}x")


## Сложность $O(n^2 \cdot d)$: откуда берётся квадрат

Основные операции Scaled Dot-Product Attention:

1. **Умножение Q и K^T:** $(n \times d) \cdot (d \times n) = O(n^2 \cdot d)$
   - Каждый из $n^2$ элементов матрицы внимания требует $d$ умножений
   - Это доминирующий член при $n \gg d$

2. **Softmax:** $O(n^2)$ — поэлементная операция, не доминирует

3. **Умножение A и V:** $(n \times n) \cdot (n \times d) = O(n^2 \cdot d)$
   - Второй квадратичный член

**Итог:** $O(n^2 \cdot d)$ — квадратичная сложность по длине последовательности.

**Почему это проблема:** при $n = 4096$ (типичный LLM) матрица внимания занимает $4096^2 \times 4$ байт $= 64$ МБ на голову. При 32 головах это уже 2 ГБ только на attention scores.

**Пути решения:**
- Sparse attention (только локальные паттерны)
- Linear attention (замена softmax на ядерный трюк)
- Flash Attention (аппаратно-оптимизированная реализация без материализации матрицы $n \times n$)


## 4.7 Визуализация: влияние масштабирования на softmax

Демонстрируем, почему деление на $\sqrt{d_k}$ критично.

In [ ]:
log.debug("Visualizing effect of scaling on softmax distribution")

# For dims [1, 8, 64, 256], compare softmax output with and without scaling
dims = [1, 8, 64, 256]
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for ax, d_k in zip(axes.flat, dims):
    Q = torch.randn(1, 10, d_k)
    K = torch.randn(1, 10, d_k)
    
    scores = Q @ K.transpose(-2, -1)
    scores_scaled = scores / math.sqrt(d_k)
    
    probs = F.softmax(scores, dim=-1)[0, 0].detach().numpy()
    probs_scaled = F.softmax(scores_scaled, dim=-1)[0, 0].detach().numpy()
    
    # Without scaling: peaky distribution (one close to 1, rest near 0)
    # With scaling: more uniform, healthier gradient flow
    ax.hist(probs, bins=20, alpha=0.5, label='Without scaling')
    ax.hist(probs_scaled, bins=20, alpha=0.5, label='With scaling')
    ax.set_title(f'd_k = {d_k}')
    ax.set_xlabel('Attention weight')
    ax.set_ylabel('Frequency')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    log.debug("d_k=%d: entropy unscaled=%.3f, scaled=%.3f",
              d_k,
              -((probs+1e-10)*np.log(probs+1e-10)).sum(),
              -((probs_scaled+1e-10)*np.log(probs_scaled+1e-10)).sum())

plt.suptitle('Effect of Scaling on Softmax Distribution', fontsize=14)
plt.tight_layout()
plt.show()
log.info("Scaling effect visualization complete")


In [ ]:
# === Summary ===
print("=== Scaled Dot-Product Attention complete ===")
print("Topics covered:")
print("  - Scaled Dot-Product Attention from scratch")
print("  - Q, K, V intuition and mathematics")
print("  - Attention heatmap visualization")
print("  - Padding mask for variable-length sequences")
print("  - Pattern matching demo on synthetic data")
print("  - Comparison with F.scaled_dot_product_attention")
print("  - Performance benchmark")
print("  - Why scaling by sqrt(d_k) matters")
# Complexity: O(n^2 * d) — quadratic in sequence length.
# This is the main bottleneck for Transformer long-context scaling.
log.info("Attention notebook complete")


📚 **Полезные ссылки:**
- [Attention Is All You Need (Vaswani et al., 2017)](https://arxiv.org/abs/1706.03762)
- [PyTorch: scaled_dot_product_attention](https://pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html)
